# Hudi vector index — structure & inspection

This notebook helps you **inspect how a vector (IVF-style) index is represented** in Hudi: metadata table (MDT) partitions, Avro payload fields, and the on-disk layout under `gs://` or local paths.

**Prerequisites**

- **Spark** (local or Spark Connect) with Hudi + GCS if you use `gs://`
- Optional: `gsutil` or Google ADC for listing / `cat` on GCS without Spark Hadoop FS
- Optional: `numpy` for centroid byte decoding

Set paths via environment variables or edit the config cell below.

## 1. Logical index structure (IVF in MDT)

Hudi’s vector index metadata is modeled as **IVF-style** records in the **metadata table** (when that partition is enabled and populated):

| Concept | Role |
|--------|------|
| **Index partition** | One MDT partition per logical index, path prefix `vector_index_<name>` (see `HoodieTableMetadataUtil.PARTITION_NAME_VECTOR_INDEX_PREFIX`). |
| **Centroid table** | A single special record whose key is **`__centroids__`**. `centroidBytes` holds **K × D × 4** bytes (**float32 little-endian**), row-major cluster centroids. |
| **Assignments** | Other records map **record key → cluster id** (`clusterId` ≥ 0). Tombstones use `isDeleted`. |

Avro shape (see `HoodieMetadata.avsc` → `HoodieVectorIndexInfo`):

- `clusterId` (int): cluster for this row key; **-1** for the centroid-dump record.
- `centroidBytes` (bytes | null): non-null **only** for key `__centroids__`.
- `isDeleted` (bool): tombstone for assignments.

Your **data table** still stores the **VECTOR(dim)** column on records (Parquet + `hudi_type` metadata); the MDT index is a **separate** acceleration structure.

## 2. Typical on-disk layout

```text
<table_base>/
  .hoodie/
    hoodie.properties          # lists enabled MDT partitions, table props
    timeline/...
  <data partition dirs>/       # contains columnar files with VECTOR column
  ...

# If metadata table is enabled:
<table_base>/.hoodie/metadata/
  .hoodie/
    hoodie.properties
    timeline/...
  vector_index_<idxName>/      # vector index MDT partition (when built)
    <file groups>.parquet / log ...
```

Use the cells below to **list** paths and **print** `hoodie.properties` for your table.

### Diagram (IVF records in MDT)

```mermaid
flowchart LR
  subgraph DATA["Data table"]
    V["embedding column\n(Parquet + hudi_type=VECTOR(dim))"]
  end
  subgraph MDT["MDT partition vector_index_*"]
    CENT["key = __centroids__\nclusterId = -1\ncentroidBytes = K×D×4 float32 LE"]
    ASG["key = record key\nclusterId = 0..K-1\nisDeleted optional"]
  end
  V -.->|indexed by| ASG
  CENT --- ASG
```

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path
from typing import List, Optional

# --- Edit or export env vars ---
# GCS base used in your tests:
TABLE_BASE = os.environ.get("HOODIE_GCS_VECTOR_BASE", "gs://mystique_qa/hudi-vector").rstrip("/")
# Point at a concrete Hudi table path (the folder containing .hoodie/)
TABLE_PATH = os.environ.get("HOODIE_VECTOR_TABLE_PATH", TABLE_BASE + "/vector_connect_it_demo").rstrip("/")

# Spark Connect (optional — leave None for gsutil-only inspection)
SPARK_REMOTE = "sc://lh-benchmark-hudi-vector.sample-app.spark.apps.useast4-dev-gke-st-spark00.k8s.us.walmart.net:443/;use_ssl=true"


print("TABLE_PATH =", TABLE_PATH)
print("SPARK_REMOTE =", SPARK_REMOTE or "(not set — Spark cells will be skipped)")

TABLE_PATH = gs://mystique_qa/hudi-vector/vector_connect_it_demo
SPARK_REMOTE = sc://lh-benchmark-hudi-vector.sample-app.spark.apps.useast4-dev-gke-st-spark00.k8s.us.walmart.net:443/;use_ssl=true


## 3. Live cluster note

From the driver pod you inspected:

- `/opt/hudi/hudi-spark4.0-bundle_2.13-1.2.0-vector-SNAPSHOT.jar`
- `/opt/hudi/hudi-azure-bundle-1.2.0-SNAPSHOT.jar`

That confirms the **Hudi jar exists on the driver filesystem**. The next question is whether the **running Spark Connect session** is actually using Hudi-related configs and classpath entries. The next cell prints the live session values before any table read.

In [ ]:
# Live Spark Connect config probe
# This is the first cell to run before any Hudi read/write.

WALMART_CA_CERT = os.path.expanduser("/Users/r0c0ezv/lakehouse/sample-app-root-ca.crt")
if os.path.exists(WALMART_CA_CERT):
    os.environ["GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"] = WALMART_CA_CERT
    os.environ["SSL_CERT_FILE"] = WALMART_CA_CERT

LIVE_CONFIG_KEYS = [
    "spark.serializer",
    "spark.kryo.registrator",
    "spark.sql.extensions",
    "spark.sql.catalog.spark_catalog",
    "spark.jars",
    "spark.driver.extraClassPath",
    "spark.executor.extraClassPath",
    "spark.hadoop.fs.gs.impl",
    "spark.hadoop.fs.AbstractFileSystem.gs.impl",
]

if SPARK_REMOTE:
    from pyspark.sql import SparkSession

    spark = SparkSession.builder.remote(SPARK_REMOTE).getOrCreate()
    print("Spark version:", spark.version)
    print("Remote:", SPARK_REMOTE)
    print()
    print("=== spark.conf.get(...) ===")
    for key in LIVE_CONFIG_KEYS:
        try:
            value = spark.conf.get(key)
        except Exception as e:
            value = f"<unset: {type(e).__name__}: {e}>"
        print(f"{key} = {value}")

    print()
    print("=== spark.sql('SET -v') filtered ===")
    try:
        rows = spark.sql("SET -v").collect()
        matches = []
        for row in rows:
            key = getattr(row, "key", None)
            if key and (
                key.startswith("spark.serializer")
                or key.startswith("spark.kryo")
                or key.startswith("spark.sql.extensions")
                or key.startswith("spark.sql.catalog.spark_catalog")
                or key.startswith("spark.jars")
                or key.startswith("spark.driver.extraClassPath")
                or key.startswith("spark.executor.extraClassPath")
                or key.startswith("spark.hadoop.fs.gs")
                or key.startswith("spark.hadoop.fs.AbstractFileSystem.gs")
            ):
                matches.append(row)
        for row in matches:
            print(f"{row.key} = {row.value}")
    except Exception as e:
        print("SET -v failed:", e)
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

Spark version: 4.0.1
Remote: sc://lh-benchmark-hudi-vector.sample-app.spark.apps.useast4-dev-gke-st-spark00.k8s.us.walmart.net:443/;use_ssl=true

=== spark.conf.get(...) ===
spark.serializer = org.apache.spark.serializer.KryoSerializer
spark.kryo.registrator = org.apache.spark.HoodieSparkKryoRegistrar
spark.sql.extensions = org.apache.spark.sql.hudi.HoodieSparkSessionExtension
spark.sql.catalog.spark_catalog = org.apache.spark.sql.hudi.catalog.HoodieCatalog
spark.jars = file:/tmp/spark-c9c704d5-1032-4c11-a33e-88f691641484/spark-sql-kafka-0-10_2.13-4.0.1.jar,file:/tmp/spark-c9c704d5-1032-4c11-a33e-88f691641484/spark-token-provider-kafka-0-10_2.13-4.0.1.jar,file:/tmp/spark-c9c704d5-1032-4c11-a33e-88f691641484/kafka-clients-3.7.1.jar,file:/tmp/spark-c9c704d5-1032-4c11-a33e-88f691641484/commons-pool2-2.12.0.jar,file:/tmp/.ivy2.5.2/jars/com.google.cloud.bigdataoss_gcs-connector-hadoop3-2.2.26-shaded.jar,file:/tmp/.ivy2.5.2/jars/org.apache.hadoop_hadoop-azure-3.4.1.jar,file:/tmp/.ivy2.5.2/ja

In [ ]:
def _run_cmd(args: List[str], timeout: int = 120) -> str:
    exe = args[0]
    if shutil.which(exe) is None:
        raise FileNotFoundError(f"executable not on PATH: {exe}")
    p = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    if p.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(args)}\n{p.stderr}")
    return p.stdout


def gsutil_ls(uri: str, recursive: bool = False, max_lines: Optional[int] = 200) -> str:
    """List GCS prefix using gsutil (must be on PATH)."""
    cmd = ["gsutil", "ls"]
    if recursive:
        cmd.append("-r")
    cmd.append(uri)
    out = _run_cmd(cmd)
    lines = out.splitlines()
    if max_lines is not None and len(lines) > max_lines:
        lines = lines[:max_lines] + [f"... ({len(out.splitlines()) - max_lines} more lines truncated)"]
    return "\n".join(lines)


def gsutil_cat(uri: str) -> str:
    return _run_cmd(["gsutil", "cat", uri])


def print_hoodie_properties(table_path: str) -> None:
    hp = f"{table_path}/.hoodie/hoodie.properties"
    print(f"--- {hp} ---")
    try:
        if table_path.startswith("gs://"):
            print(gsutil_cat(hp))
        else:
            print(Path(hp).read_text())
    except Exception as e:
        print(f"(could not read: {e})")


def print_metadata_tree(table_path: str) -> None:
    m = f"{table_path}/.hoodie/metadata"
    print(f"--- listing {m} (recursive, truncated) ---")
    try:
        if table_path.startswith("gs://"):
            print(gsutil_ls(m, recursive=True, max_lines=300))
        else:
            root = Path(m)
            if not root.exists():
                print(f"(path does not exist: {root})")
                return
            paths = sorted(root.rglob("*"))[:300]
            for p in paths:
                print(p)
    except Exception as e:
        print(f"(could not list: {e})")

In [ ]:
# Table-level properties (partitions, metadata enablement)
print_hoodie_properties(TABLE_PATH)

In [ ]:
# If MDT exists under the table, show files (look for vector_index_* folders)
print_metadata_tree(TABLE_PATH)

## 3. What to look for in `hoodie.properties`

After running the print cell above, search the output for:

- **`hoodie.metadata.enable`** — metadata table must be enabled for MDT index partitions.
- **`hoodie.table.metadata.partitions`** — comma-separated list; look for **`vector_index_...`** entries when the vector index partition is registered.
- **`hoodie.table.metadata.partitions.inflight`** — partitions still building.

If you only see **`files`**, **`column_stats`**, etc., but no **`vector_index_`**, the IVF index partition may not be initialized yet (depends on your Hudi version and write pipeline).

**Data plane** (VECTOR column) still appears in the Parquet schema as **`ArrayType(FloatType)`** with Spark field metadata **`hudi_type=VECTOR(dim)`** — that is independent of whether the MDT vector index partition exists.

In [ ]:
# Optional: decode centroidBytes layout (K clusters × D dims × float32)
# Use when you have raw bytes (e.g. from debugging MDT); K = len(buf) // (D * 4).

import numpy as np


def decode_centroids_le(buf: bytes, dim: int) -> np.ndarray:
    """Return array of shape (K, dim) for little-endian float32 centroid blob."""
    n_floats = len(buf) // 4
    if n_floats * 4 != len(buf):
        raise ValueError("byte length must be multiple of 4")
    if n_floats % dim != 0:
        raise ValueError(f"n_floats={n_floats} not divisible by dim={dim}")
    flat = np.frombuffer(buf, dtype="<f4")
    k = n_floats // dim
    return flat.reshape(k, dim)


# Example: 3 clusters, 8 dimensions → 3 * 8 * 4 = 96 bytes
_demo = np.arange(3 * 8, dtype="<f4").tobytes()
print("demo centroids shape:", decode_centroids_le(_demo, dim=8).shape)
print(decode_centroids_le(_demo, dim=8))

## 5. Optional: Spark — read an existing Hudi table

Run the **live config probe** cell above first. This cell does **not** deserialize MDT Avro payloads, but it confirms the **embedding** column and **`hudi_type`** metadata on the dataset.

Requires: `pyspark`, cluster with Hudi + GCS, and a real Hudi table at `TABLE_PATH`. If `TABLE_PATH` does not exist yet, run the **build sample table** cell below first.

In [ ]:
HOODIE_TYPE_KEY = "hudi_type"  # HoodieSchema.TYPE_METADATA_FIELD
WALMART_CA_CERT = os.path.expanduser("/Users/r0c0ezv/lakehouse/sample-app-root-ca.crt")
if os.path.exists(WALMART_CA_CERT):
    os.environ["GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"] = WALMART_CA_CERT
    os.environ["SSL_CERT_FILE"] = WALMART_CA_CERT
    print(f"SSL cert configured: {WALMART_CA_CERT}")
else:
    print(f"SSL cert not found at {WALMART_CA_CERT}")

if SPARK_REMOTE:
    from pyspark.sql import SparkSession

    spark = SparkSession.builder.remote(SPARK_REMOTE).getOrCreate()
    try:
        print("Attempting Hudi read from:", TABLE_PATH)
        df = spark.read.format("hudi").load(TABLE_PATH)
        print("rows:", df.count())
        df.printSchema()
        for f in df.schema.fields:
            if f.name == "embedding":
                print("embedding metadata:", dict(f.metadata))
    except Exception as e:
        print("Hudi read failed:", type(e).__name__, e)
        print("Hint: verify the live config probe shows Hudi extensions/catalog, and that TABLE_PATH points to an existing Hudi table.")
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

SSL cert configured: /Users/r0c0ezv/lakehouse/sample-app-root-ca.crt
Attempting Hudi read from: gs://mystique_qa/hudi-vector/vector_connect_it_demo
Hudi read failed: UnknownException (java.io.FileNotFoundException) File not found: gs://mystique_qa/hudi-vector/vector_connect_it_demo

JVM stacktrace:
java.io.FileNotFoundException
	at com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystemBase.lambda$getFileStatus$10(GoogleHadoopFileSystemBase.java:1085)
	at com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics.trackDuration(GhfsStorageStatistics.java:102)
	at com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystemBase.getFileStatus(GoogleHadoopFileSystemBase.java:1070)
	at org.apache.hudi.storage.hadoop.HoodieHadoopStorage.getPathInfo(HoodieHadoopStorage.java:170)
	at org.apache.hudi.common.util.TablePathUtils.getTablePath(TablePathUtils.java:58)
	at org.apache.hudi.DataSourceUtils.getTablePath(DataSourceUtils.java:80)
	at org.apache.hudi.DefaultSource.createRelation(DefaultSource.scala:114)


## 6. Build a sample Hudi vector table

This cell creates a small deterministic Hudi table at `TABLE_PATH` so the later read and search cells have a concrete dataset to work with.

Dataset shape:

- 9 rows
- 3 logical clusters: `west`, `mid`, `east`
- `embedding` stored as `ARRAY<FLOAT>` with Spark field metadata `hudi_type=VECTOR(8)`
- write mode: `COPY_ON_WRITE`

After this runs, rerun the `hoodie.properties` and metadata tree cells above to inspect the table layout.

In [ ]:
# Build a small Hudi table with deterministic VECTOR(8) embeddings.

import json

HOODIE_TYPE_KEY = "hudi_type"
WALMART_CA_CERT = os.path.expanduser("/Users/r0c0ezv/lakehouse/sample-app-root-ca.crt")
if os.path.exists(WALMART_CA_CERT):
    os.environ["GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"] = WALMART_CA_CERT
    os.environ["SSL_CERT_FILE"] = WALMART_CA_CERT

if SPARK_REMOTE:
    from pyspark.sql import SparkSession
    from pyspark.sql.types import ArrayType, FloatType, StringType, StructField, StructType

    dim = 8
    schema = StructType([
        StructField("id", StringType(), False),
        StructField("label", StringType(), False),
        StructField("embedding", ArrayType(FloatType(), False), False, metadata={HOODIE_TYPE_KEY: f"VECTOR({dim})"}),
    ])

    def vector_write_schema_json(d: int) -> str:
        return json.dumps({
            "type": "record",
            "name": "vector_connect_record",
            "namespace": "hoodie.vector.connect",
            "fields": [
                {"name": "id", "type": "string"},
                {"name": "label", "type": "string"},
                {
                    "name": "embedding",
                    "type": {
                        "type": "fixed",
                        "name": f"vector_float_{d}",
                        "size": d * 4,
                        "logicalType": "vector",
                        "dimension": d,
                        "elementType": "FLOAT",
                        "storageBacking": "FIXED_BYTES",
                    },
                },
            ],
        })

    def vec(x: float):
        return [float(x)] + [0.0] * (dim - 1)

    rows = [
        ("west-0", "west", vec(0.0)),
        ("west-1", "west", vec(0.4)),
        ("west-2", "west", vec(0.8)),
        ("mid-0", "mid", vec(10.0)),
        ("mid-1", "mid", vec(10.3)),
        ("mid-2", "mid", vec(9.7)),
        ("east-0", "east", vec(20.0)),
        ("east-1", "east", vec(20.4)),
        ("east-2", "east", vec(19.6)),
    ]

    spark = SparkSession.builder.remote(SPARK_REMOTE).getOrCreate()
    try:
        df = spark.createDataFrame(rows, schema=schema)
        (
            df.write.format("hudi")
            .mode("overwrite")
            .option("hoodie.datasource.write.recordkey.field", "id")
            .option("hoodie.datasource.write.precombine.field", "id")
            .option("hoodie.datasource.write.table.name", "vector_connect_it_demo")
            .option("hoodie.datasource.write.table.type", "COPY_ON_WRITE")
            .option("hoodie.write.schema", vector_write_schema_json(dim))
            .option("hoodie.metadata.enable", "true")
            .option("path", TABLE_PATH)
            .save()
        )
        print(f"Built sample Hudi vector table at: {TABLE_PATH}")
        print("Rows written:", len(rows))
        print("Labels:", sorted({r[1] for r in rows}))
    except Exception as e:
        print("Build failed:", type(e).__name__, e)
        print("Hint: rerun the live config probe first and confirm the session is Hudi-enabled.")
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

Built sample Hudi vector table at: gs://mystique_qa/hudi-vector/vector_connect_it_demo
Rows written: 9
Labels: ['east', 'mid', 'west']


## 7. Explicit vector-index build attempt

This cell rewrites the sample table using the **RFC draft** vector-index writer options:

- `hoodie.metadata.vector.index.enable`
- `hoodie.metadata.vector.index.column`
- `hoodie.metadata.vector.index.dimension`
- `hoodie.metadata.vector.index.metric`
- `hoodie.metadata.vector.index.algorithm`
- `hoodie.metadata.vector.index.num_clusters`

Important: these keys are documented in `rfc/rfc-103/rfc-103.md`, but they are **not currently discoverable in the implementation code** in this branch. So this cell may do one of three things:

1. fail fast,
2. succeed but ignore the options,
3. succeed and actually create vector-index MDT artifacts.

The next verification cell checks the real artifact: a `vector_index_<name>` metadata partition.

In [ ]:
# Attempt an explicit vector-index build using the RFC draft datasource options.

import json

VECTOR_INDEX_NAME = "embedding_idx"
VECTOR_INDEX_PARTITION = f"vector_index_{VECTOR_INDEX_NAME}"

WALMART_CA_CERT = os.path.expanduser("/Users/r0c0ezv/lakehouse/sample-app-root-ca.crt")
if os.path.exists(WALMART_CA_CERT):
    os.environ["GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"] = WALMART_CA_CERT
    os.environ["SSL_CERT_FILE"] = WALMART_CA_CERT

if SPARK_REMOTE:
    from pyspark.sql import SparkSession
    from pyspark.sql.types import ArrayType, FloatType, StringType, StructField, StructType

    dim = 8
    schema = StructType([
        StructField("id", StringType(), False),
        StructField("label", StringType(), False),
        StructField("embedding", ArrayType(FloatType(), False), False, metadata={"hudi_type": f"VECTOR({dim})"}),
    ])

    def vector_write_schema_json(d: int) -> str:
        return json.dumps({
            "type": "record",
            "name": "vector_connect_record",
            "namespace": "hoodie.vector.connect",
            "fields": [
                {"name": "id", "type": "string"},
                {"name": "label", "type": "string"},
                {
                    "name": "embedding",
                    "type": {
                        "type": "fixed",
                        "name": f"vector_float_{d}",
                        "size": d * 4,
                        "logicalType": "vector",
                        "dimension": d,
                        "elementType": "FLOAT",
                        "storageBacking": "FIXED_BYTES",
                    },
                },
            ],
        })

    def vec(x: float):
        return [float(x)] + [0.0] * (dim - 1)

    rows = [
        ("west-0", "west", vec(0.0)),
        ("west-1", "west", vec(0.4)),
        ("west-2", "west", vec(0.8)),
        ("mid-0", "mid", vec(10.0)),
        ("mid-1", "mid", vec(10.3)),
        ("mid-2", "mid", vec(9.7)),
        ("east-0", "east", vec(20.0)),
        ("east-1", "east", vec(20.4)),
        ("east-2", "east", vec(19.6)),
    ]

    spark = SparkSession.builder.remote(SPARK_REMOTE).getOrCreate()
    try:
        df = spark.createDataFrame(rows, schema=schema)
        (
            df.write.format("hudi")
            .mode("overwrite")
            .option("hoodie.datasource.write.recordkey.field", "id")
            .option("hoodie.datasource.write.precombine.field", "id")
            .option("hoodie.datasource.write.table.name", "vector_connect_it_demo")
            .option("hoodie.datasource.write.table.type", "COPY_ON_WRITE")
            .option("hoodie.write.schema", vector_write_schema_json(dim))
            .option("hoodie.metadata.enable", "true")
            .option("hoodie.metadata.vector.index.enable", "true")
            .option("hoodie.metadata.vector.index.name", VECTOR_INDEX_NAME)
            .option("hoodie.metadata.vector.index.column", "embedding")
            .option("hoodie.metadata.vector.index.dimension", str(dim))
            .option("hoodie.metadata.vector.index.metric", "l2")
            .option("hoodie.metadata.vector.index.algorithm", "ivfflat")
            .option("hoodie.metadata.vector.index.num_clusters", "3")
            .option("path", TABLE_PATH)
            .save()
        )
        print("Explicit vector-index build attempt finished.")
        print("Requested index name:", VECTOR_INDEX_NAME)
        print("Expected MDT partition:", VECTOR_INDEX_PARTITION)
    except Exception as e:
        print("Explicit vector-index build failed:", type(e).__name__, e)
        print("This usually means the session is not Hudi-enabled, or the branch does not yet implement the RFC vector-index writer options.")
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

## 8. Verify vector-index artifacts

This is the real test for the explicit build above.

It checks two things:

- whether `hoodie.properties` declares a `vector_index_<name>` metadata partition
- whether files exist under `.hoodie/metadata/vector_index_<name>/`

If both checks fail, then the RFC options were ignored or the explicit vector-index builder is not implemented in this branch.

## 9. MDT vector metadata samples

This section prints concrete rows from `hudi_metadata(...)` for the vector index partition:

- the centroid record
- the quantizer record (`IVF_RABITQ` seed / code width / normalization mode)
- assignment samples (`record_key -> cluster_id + file_group_id + partition_path`)
- `fg_mapping` samples (`cluster_id + partition_path -> file_group_ids`)

These are the most direct way to validate that the index build wrote the expected MDT structures.

In [ ]:
VECTOR_RECORD_TYPE = 8
VECTOR_FG_KEY_PREFIX = "__fg__/"
VECTOR_CENTROIDS_KEY = "__centroids__"
VECTOR_QUANTIZER_KEY = "__quantizer__"

if SPARK_REMOTE:
    from pyspark.sql import SparkSession

    spark = SparkSession.builder.remote(SPARK_REMOTE).getOrCreate()

    centroid_df = spark.sql(
        f"""
        SELECT
          key,
          VectorIndexMetadata.clusterId AS cluster_id,
          length(VectorIndexMetadata.centroidBytes) AS centroid_bytes
        FROM hudi_metadata('{TABLE_PATH}')
        WHERE type = {VECTOR_RECORD_TYPE}
          AND key = '{VECTOR_CENTROIDS_KEY}'
        """
    )
    print("Centroid record:")
    centroid_df.show(truncate=False)

    quantizer_df = spark.sql(
        f"""
        SELECT
          key,
          VectorIndexMetadata.quantizerType AS quantizer_type,
          VectorIndexMetadata.quantizedCodeBytes AS quantized_code_bytes,
          VectorIndexMetadata.randomSeed AS random_seed,
          VectorIndexMetadata.assumeNormalized AS assume_normalized
        FROM hudi_metadata('{TABLE_PATH}')
        WHERE type = {VECTOR_RECORD_TYPE}
          AND key = '{VECTOR_QUANTIZER_KEY}'
        """
    )
    print("Quantizer metadata:")
    quantizer_df.show(truncate=False)

    assignments_df = spark.sql(
        f"""
        SELECT
          key,
          VectorIndexMetadata.clusterId AS cluster_id,
          VectorIndexMetadata.fileGroupId AS file_group_id,
          VectorIndexMetadata.partitionPath AS partition_path,
          VectorIndexMetadata.isDeleted AS is_deleted
        FROM hudi_metadata('{TABLE_PATH}')
        WHERE type = {VECTOR_RECORD_TYPE}
          AND key <> '{VECTOR_CENTROIDS_KEY}'
          AND key <> '{VECTOR_QUANTIZER_KEY}'
          AND key NOT LIKE '{VECTOR_FG_KEY_PREFIX}%'
        ORDER BY key
        LIMIT 10
        """
    )
    print("Assignment samples:")
    assignments_df.show(truncate=False)

    fg_mapping_df = spark.sql(
        f"""
        SELECT
          key,
          VectorIndexMetadata.clusterId AS cluster_id,
          VectorIndexMetadata.partitionPath AS partition_path,
          VectorIndexMetadata.fileGroupIds AS file_group_ids,
          VectorIndexMetadata.vectorCount AS vector_count,
          VectorIndexMetadata.lastUpdatedTs AS last_updated_ts
        FROM hudi_metadata('{TABLE_PATH}')
        WHERE type = {VECTOR_RECORD_TYPE}
          AND key LIKE '{VECTOR_FG_KEY_PREFIX}%'
        ORDER BY key
        LIMIT 10
        """
    )
    print("fg_mapping samples:")
    fg_mapping_df.show(truncate=False)
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

In [ ]:
# Verify whether the explicit vector-index build actually produced MDT artifacts.

WALMART_CA_CERT = os.path.expanduser("/Users/r0c0ezv/lakehouse/sample-app-root-ca.crt")
if os.path.exists(WALMART_CA_CERT):
    os.environ["GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"] = WALMART_CA_CERT
    os.environ["SSL_CERT_FILE"] = WALMART_CA_CERT

if SPARK_REMOTE:
    from pyspark.sql import SparkSession

    spark = SparkSession.builder.remote(SPARK_REMOTE).getOrCreate()
    properties_path = f"{TABLE_PATH}/.hoodie/hoodie.properties"
    metadata_glob = f"{TABLE_PATH}/.hoodie/metadata/{VECTOR_INDEX_PARTITION}/*"

    partition_declared = False
    artifact_count = 0

    try:
        props_df = spark.read.text(properties_path)
        prop_lines = [r.value for r in props_df.collect()]
        interesting = [line for line in prop_lines if "metadata" in line or "vector" in line]
        print("Relevant hoodie.properties lines:")
        for line in interesting:
            print(line)
        partition_declared = any(VECTOR_INDEX_PARTITION in line for line in prop_lines)
    except Exception as e:
        print("Could not read hoodie.properties via Spark:", type(e).__name__, e)

    try:
        files_df = spark.read.format("binaryFile").load(metadata_glob)
        artifact_count = files_df.count()
        print(f"Files under {metadata_glob}: {artifact_count}")
    except Exception as e:
        print("Could not list vector-index metadata artifacts via Spark:", type(e).__name__, e)

    print("partition_declared =", partition_declared)
    print("artifact_count =", artifact_count)

    assert partition_declared or artifact_count > 0, (
        f"No explicit vector-index artifacts found for {VECTOR_INDEX_PARTITION}. "
        "The RFC option keys were likely ignored, or the builder path is not implemented in this branch."
    )
    print("Vector-index artifact verification passed.")
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

NameError: name 'VECTOR_INDEX_PARTITION' is not defined

## 9. Brute-force search baseline

This is a small **brute-force nearest-neighbor** search over the stored embeddings. It is useful as a correctness baseline after the build.

This cell does **not** prove the metadata-table vector index was used. The explicit vector-index test in this notebook is the MDT artifact check above.

Query vector: near the `mid` cluster.

Expected result:

- top-1 label is `mid`
- top-3 results should all come from the `mid` cluster in this synthetic dataset

In [ ]:
import math

WALMART_CA_CERT = os.path.expanduser("/Users/r0c0ezv/lakehouse/sample-app-root-ca.crt")
if os.path.exists(WALMART_CA_CERT):
    os.environ["GRPC_DEFAULT_SSL_ROOTS_FILE_PATH"] = WALMART_CA_CERT
    os.environ["SSL_CERT_FILE"] = WALMART_CA_CERT


def l2(a, b):
    return math.sqrt(sum((float(x) - float(y)) ** 2 for x, y in zip(a, b)))


if SPARK_REMOTE:
    from pyspark.sql import SparkSession

    spark = SparkSession.builder.remote(SPARK_REMOTE).getOrCreate()
    try:
        df = spark.read.format("hudi").load(TABLE_PATH)
        rows = df.select("id", "label", "embedding").collect()
        query = [10.1] + [0.0] * 7
        scored = sorted(
            [(row.id, row.label, l2(row.embedding, query)) for row in rows],
            key=lambda x: x[2],
        )
        top_k = scored[:3]
        print("Query vector:", query)
        print("Top-3 nearest rows:")
        for rank, (row_id, label, distance) in enumerate(top_k, start=1):
            print(f"{rank}. id={row_id}, label={label}, l2={distance:.4f}")

        assert top_k[0][1] == "mid", f"expected top-1 label mid, got {top_k[0][1]}"
        assert all(label == "mid" for _, label, _ in top_k), f"expected all top-3 labels to be mid, got {top_k}"
        print("Brute-force vector search baseline passed.")
    except Exception as e:
        print("Search failed:", type(e).__name__, e)
        print("Hint: build the sample table first, then rerun this cell.")
else:
    print("Set SPARK_REMOTE in the config cell (or env) to run this block.")

## 10. MDT-backed vector-pruned read path

This is the current **testable query surface** for Phase 2 coarse pruning.

Use regular Hudi read options:

- `hoodie.datasource.read.vector.index.name`
- `hoodie.datasource.read.vector.query.vector`
- `hoodie.datasource.read.vector.query.nprobes`

These options let `HoodieFileIndex` load centroids + `fg_mapping` from MDT and prune candidate file groups before the scan. This is intentionally separate from the future SQL function surface.

The code cell below also includes a small timing harness so you can compare full-table scan time with the current MDT-pruned candidate scan after deployment.

In [ ]:
# Exercise the MDT-backed vector pruning read path.
# This validates the current file-index hook and, when hidden columns are present,
# benchmarks a Spark-first RaBitQ approximate scan plus exact rerank.

import time

VECTOR_INDEX_NAME = os.environ.get("HOODIE_VECTOR_INDEX_NAME", "vec_idx")
VECTOR_NPROBES = os.environ.get("HOODIE_VECTOR_NPROBES", "1")
BENCHMARK_ITERATIONS = int(os.environ.get("HOODIE_VECTOR_BENCHMARK_ITERATIONS", "3"))
RABITQ_APPROX_DISTANCE_FN = "hudi_rabitq_distance"
RABITQ_APPROX_DISTANCE_UDF_CLASS = "org.apache.hudi.sql.RaBitQApproxDistanceUDF"
RABITQ_RERANK_LIMIT = int(os.environ.get("HOODIE_VECTOR_RABITQ_RERANK_LIMIT", "32"))


def logical_index_name(index_name):
    return index_name.removeprefix("vector_index_")


def rabitq_binary_column(index_name):
    return f"_hudi_vec_{logical_index_name(index_name)}_binary_code"


def rabitq_scalar_column(index_name):
    return f"_hudi_vec_{logical_index_name(index_name)}_scalar"


def register_rabitq_distance_udf(spark_session):
    try:
        spark_session.udf.registerJavaFunction(
            RABITQ_APPROX_DISTANCE_FN, RABITQ_APPROX_DISTANCE_UDF_CLASS, "float"
        )
        print(
            "Registered Spark SQL RaBitQ UDF:",
            f"{RABITQ_APPROX_DISTANCE_FN} -> {RABITQ_APPROX_DISTANCE_UDF_CLASS}",
        )
        return True
    except Exception as exc:
        print("Skipping RaBitQ benchmark: failed to register Java UDF:", exc)
        return False


def exact_l2_sql(vector_col, query_vector):
    query_array_sql = ", ".join(f"{float(value):.12g}D" for value in query_vector)
    return (
        "sqrt(aggregate("
        f"zip_with({vector_col}, array({query_array_sql}), "
        "(x, y) -> pow(cast(x as double) - y, 2D)), "
        "cast(0D as double), "
        "(acc, x) -> acc + x))"
    )


if "spark" not in globals():
    print("Spark session not initialized; run the Spark setup cells first.")
else:
    source_df = spark.read.format("hudi").load(TABLE_PATH)
    sample_rows = source_df.select("id", "embedding").limit(5).collect()
    if not sample_rows:
        print("No rows found in table; build/write the demo table first.")
    else:
        query_vector = sample_rows[0].embedding
        query_literal = "[" + ", ".join(f"{float(v):.6f}" for v in query_vector) + "]"
        print("Using vector-pruned read options:")
        print("  hoodie.datasource.read.vector.index.name =", VECTOR_INDEX_NAME)
        print("  hoodie.datasource.read.vector.query.vector =", query_literal)
        print("  hoodie.datasource.read.vector.query.nprobes =", VECTOR_NPROBES)

        pruned_df = (
            spark.read.format("hudi")
            .option("hoodie.datasource.read.vector.index.name", VECTOR_INDEX_NAME)
            .option("hoodie.datasource.read.vector.query.vector", query_literal)
            .option("hoodie.datasource.read.vector.query.nprobes", VECTOR_NPROBES)
            .load(TABLE_PATH)
        )

        print("\nPhysical plan:")
        pruned_df.select("id", "embedding").limit(10).explain(True)

        print("\nSample rows from vector-pruned read path:")
        display(pruned_df.select("id", "embedding").limit(5))

        rabitq_binary_col = rabitq_binary_column(VECTOR_INDEX_NAME)
        rabitq_scalar_col = rabitq_scalar_column(VECTOR_INDEX_NAME)
        available_columns = set(pruned_df.columns)
        print("\nLooking for RaBitQ hidden columns:", rabitq_binary_col, rabitq_scalar_col)
        print(
            "RaBitQ hidden column presence:",
            f"binary_code={rabitq_binary_col in available_columns},",
            f"scalar={rabitq_scalar_col in available_columns}",
        )

        def time_action(name, action, iterations=BENCHMARK_ITERATIONS):
            times = []
            last_rows = None
            last_detail = None
            for _ in range(iterations):
                t0 = time.time()
                rows, detail = action()
                times.append(time.time() - t0)
                last_rows = rows
                last_detail = detail
            avg = sum(times) / len(times)
            print(f"{name}: avg={avg:.3f}s over {iterations} runs, rows={last_rows}, detail={last_detail}")
            return avg, last_rows, last_detail

        def full_scan_action():
            rows = spark.read.format("hudi").load(TABLE_PATH).select("id", "embedding").count()
            return rows, None

        def pruned_scan_action():
            rows = (
                spark.read.format("hudi")
                .option("hoodie.datasource.read.vector.index.name", VECTOR_INDEX_NAME)
                .option("hoodie.datasource.read.vector.query.vector", query_literal)
                .option("hoodie.datasource.read.vector.query.nprobes", VECTOR_NPROBES)
                .load(TABLE_PATH)
                .select("id", "embedding")
                .count()
            )
            return rows, None

        print("\nBenchmark:")
        full_avg, full_rows, _ = time_action("full scan", full_scan_action)
        pruned_avg, pruned_rows, _ = time_action("mdt-pruned scan", pruned_scan_action)
        speedup = full_avg / pruned_avg if pruned_avg > 0 else float("inf")
        print(
            f"Benchmark summary: full_scan_avg={full_avg:.3f}s, pruned_scan_avg={pruned_avg:.3f}s, "
            f"speedup={speedup:.2f}x, full_rows={full_rows}, pruned_rows={pruned_rows}"
        )

        metadata_df = spark.sql(
            f"""
            SELECT
              VectorIndexMetadata.quantizerType AS quantizer_type,
              VectorIndexMetadata.randomSeed AS random_seed,
              VectorIndexMetadata.assumeNormalized AS assume_normalized
            FROM hudi_metadata('{TABLE_PATH}')
            WHERE type = 8 AND key = '__quantizer__'
            """
        )
        quantizer_rows = metadata_df.collect()
        quantizer_row = quantizer_rows[0] if quantizer_rows else None

        if (
            quantizer_row
            and quantizer_row.quantizer_type == "IVF_RABITQ"
            and rabitq_binary_col in available_columns
            and register_rabitq_distance_udf(spark)
        ):
            scalar_sql = rabitq_scalar_col if rabitq_scalar_col in available_columns else "CAST(NULL AS FLOAT)"
            approx_distance_sql = (
                f"{RABITQ_APPROX_DISTANCE_FN}("
                f"{rabitq_binary_col}, "
                f"{scalar_sql}, "
                f"'{query_literal}', "
                f"{len(query_vector)}, "
                f"{int(quantizer_row.random_seed)}L, "
                f"{str(bool(quantizer_row.assume_normalized)).lower()})"
            )
            exact_distance_sql = exact_l2_sql("embedding", query_vector)
            [".232","1.232","2.232","3.232","4.232","5.232","6.232","7.232","8.232","9.232","10.232"]
            ["-1","1",""]
            def rabitq_scan_action():
                scalar_cols = [rabitq_scalar_col] if rabitq_scalar_col in available_columns else []
                candidate_df = (
                    spark.read.format("hudi")
                    .option("hoodie.datasource.read.vector.index.name", VECTOR_INDEX_NAME)
                    .option("hoodie.datasource.read.vector.query.vector", query_literal)
                    .option("hoodie.datasource.read.vector.query.nprobes", VECTOR_NPROBES)
                    .load(TABLE_PATH)
                    .select("id", "embedding", rabitq_binary_col, *scalar_cols)
                    .cache()
                )
                candidate_count = candidate_df.count()
                approx_ranked = candidate_df.selectExpr(
                    "id",
                    "embedding",
                    f"{approx_distance_sql} AS approx_distance",
                )
                reranked = (
                    approx_ranked.orderBy("approx_distance")
                    .limit(RABITQ_RERANK_LIMIT)
                    .selectExpr("id", f"{exact_distance_sql} AS exact_distance")
                    .orderBy("exact_distance")
                    .limit(1)
                    .collect()
                )
                candidate_df.unpersist()
                detail = None if not reranked else f"top1_id={reranked[0].id}, exact_l2={reranked[0].exact_distance:.6g}"
                return candidate_count, detail

            rabitq_avg, rabitq_rows, rabitq_detail = time_action(
                "mdt-pruned + rabitq approx + exact rerank", rabitq_scan_action
            )
            rabitq_speedup = full_avg / rabitq_avg if rabitq_avg > 0 else float("inf")
            print(
                f"RaBitQ benchmark summary: rabitq_avg={rabitq_avg:.3f}s, "
                f"speedup_vs_full={rabitq_speedup:.2f}x, candidate_rows={rabitq_rows}, detail={rabitq_detail}"
            )
        else:
            print(
                "Skipping RaBitQ timing. This needs the new Spark jar plus materialized hidden code columns on the table."
            )